# CV Masterclass 12: The Training Pipeline

Welcome to Notebook 12. Having designed your Computer Vision architecture and selected your augmentations, you must now physically train the model.

A poorly configured training pipeline can make a state-of-the-art Vision Transformer diverge completely, while an expert pipeline can squeeze an extra 5% accuracy out of a decade-old ResNet.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"During Mixed Precision Training, gradients are stored and mathematically scaled in FP32, but standard forward/backward passes use FP16. If an FP16 minimum is approximately `6e-8`, why must we explicitly use a 'Loss Scaler' before backpropagation to prevent the entire model from decaying to zero accuracy?"*

---

## Table of Contents
1. **Optimizers: Beyond SGD:** Adaptive gradients and AdamW.
2. **Learning Rate Schedulers:** Managing the descent trajectory.
3. **Mixed Precision Training (AMP):** Doubling GPU throughput.
4. **Evaluation Metrics Reference:** Mathematical metrics for all CV tasks.


## 1. Optimizers: Beyond SGD

Standard Stochastic Gradient Descent (SGD) subtracts the average gradient from the weights. 

**Deriving Adam (Adaptive Moment Estimation):**
Adam maintains two Exponential Moving Averages (EMA):
1.  **First Moment ($m_t$):** EMA of past gradients (like Momentum).
    $$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
2.  **Second Moment ($v_t$):** EMA of past *squared* gradients.
    $$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

Because these are initialized to 0, they are biased heavily towards 0 early in training. Adam applies **Bias Correction**:
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

**The Update Rule:**
$$w_{t+1} = w_t - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

**Why does squaring gradients provide adaptive Learning Rates?**
If a feature is extremely common (e.g., green pixels for grass), its gradient is consistently large. $v_t$ (the squared gradient tracker) explodes. Since we divide by $\sqrt{v_t}$, the effective learning rate for grass becomes extremely small! 
If a feature is rare (a specific defect), $v_t$ is tiny, so the effective learning rate is multiplied heavily. It dynamically accelerates learning on rare features.

**Evolution of Optimizers:**
*   **SGD+Momentum:** Fast, but requires perfect LR tuning.
*   **Adam:** Great generically, but standard Weight Decay formulation interacts poorly with scaling.
*   **AdamW:** Strictly decouples weight decay away from the adaptive gradient scaling explicitly. (Current CV Default).
*   **Lion:** Google's new optimizer discovered by an AI. Only tracks momentum and explicitly drops scaling in favor of the algebraic `sign()` operation. Highly memory efficient.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# -----------------------------------------------------
# IMPLEMENTATION: Optimizer Convergence Comparison
# -----------------------------------------------------

class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 50)
        self.fc2 = nn.Linear(50, 1)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

# Generate synthetic deterministic regression data
torch.manual_seed(42)
X = torch.randn(200, 10)
y = X @ torch.randn(10, 1) + 2.0

def train_sim(opt_class, name, steps=100, **kwargs):
    model = DummyModel()
    optimizer = opt_class(model.parameters(), **kwargs)
    loss_fn = nn.MSELoss()
    losses = []
    
    for _ in range(steps):
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

losses_sgd = train_sim(optim.SGD, "SGD+Momentum", lr=0.1, momentum=0.9,
                        weight_decay=1e-4)
losses_adam = train_sim(optim.Adam, "Adam", lr=1e-3, weight_decay=0)
losses_adamw = train_sim(optim.AdamW, "AdamW", lr=1e-3, weight_decay=0.01)

# Note: SGD typically uses LR ~100x higher than Adam because it does
# not scale gradients adaptively. At equal LR, SGD appears to converge
# slower — this is an unfair comparison, not a fundamental difference.

plt.figure(figsize=(8, 4))
plt.plot(losses_sgd, label="SGD+Momentum", linewidth=2)
plt.plot(losses_adam, label="Adam", linewidth=2)
plt.plot(losses_adamw, label="AdamW", linewidth=2, linestyle='--')
plt.title("Optimizer Convergence on Toy Data")
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# TEST IT: The plot will generate showing Adam/AdamW converting structurally faster early.


### COMMON PITFALLS
*   **Weight Decay in Adam:** Using standard `torch.optim.Adam(weight_decay=0.01)` causes the weight decay penalty to be squashed inside the moving average. You must specifically use `optim.AdamW` to apply the mathematical $L_2$ penalty directly.


## 2. Learning Rate Schedulers

Finding the right loss valley is entirely controlled by the Learning Rate over time.

1.  **StepLR:** The classic standard (e.g., drop LR by $10\times$ every 30 epochs). Standard for ResNets.
2.  **CosineAnnealingLR:** Smooth mathematical decay following half a cosine wave down to $0.0$.
3.  **OneCycleLR:** Extremely aggressive curve. Massively ramps LR up then decays it rapidly. Ideal for super-convergence.
4.  **Warmup + Cosine:** **CRITICAL FOR VITS.** Vision Transformers initialize completely erratically. If you drop a massive generic LR on them in epoch 1, they will literally explode to `NaN` instantly. You must start at 0, slowly warm up linearly over 5-10 epochs, then Coasting-Cosine down.


In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: Visualizing Schedulers
# -----------------------------------------------------

model = nn.Linear(2, 2)

# Schedulers
opt_step = optim.Adam(model.parameters(), lr=0.1)
sch_step = optim.lr_scheduler.StepLR(opt_step, step_size=30, gamma=0.1)

opt_cos = optim.Adam(model.parameters(), lr=0.1)
sch_cos = optim.lr_scheduler.CosineAnnealingLR(opt_cos, T_max=100)

opt_one = optim.Adam(model.parameters(), lr=0.1)
sch_one = optim.lr_scheduler.OneCycleLR(opt_one, max_lr=0.1, total_steps=100)

def visualize_scheduler(scheduler, opt, epochs=100):
    lrs = []
    for _ in range(epochs):
        lrs.append(opt.param_groups[0]['lr'])
        opt.step()
        scheduler.step()
    return lrs

lr_step = visualize_scheduler(sch_step, opt_step)
lr_cos = visualize_scheduler(sch_cos, opt_cos)
lr_one = visualize_scheduler(sch_one, opt_one)

plt.figure(figsize=(8, 4))
plt.plot(lr_step, label="StepLR", lw=2)
plt.plot(lr_cos, label="CosineAnnealing", lw=2)
plt.plot(lr_one, label="OneCycle", lw=2)
plt.title("Learning Rate Evolution (100 Epochs)")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# TEST IT: The code simulates exactly how your optimizer hyperparams will alter across time.


In [ ]:
def get_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs,
                                 min_lr=1e-6):
    """
    Linear warmup for warmup_epochs, then cosine decay to min_lr.
    This is MANDATORY for Vision Transformers — without warmup, ViT
    diverges in the first epoch due to randomly initialized attention
    maps producing extreme gradient magnitudes.
    """
    def lr_lambda(current_epoch):
        if current_epoch < warmup_epochs:
            return float(current_epoch) / float(max(1, warmup_epochs))
        # Cosine decay
        progress = (current_epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        return max(min_lr / optimizer.param_groups[0]['lr'], cosine_decay)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# TEST IT
import math
vit_optimizer = optim.AdamW([torch.randn(1, requires_grad=True)], lr=1e-3)
vit_scheduler = get_warmup_cosine_scheduler(vit_optimizer, warmup_epochs=5,
                                             total_epochs=100)

vit_lrs = []
for epoch in range(100):
    vit_lrs.append(vit_optimizer.param_groups[0]['lr'])
    vit_optimizer.step()
    vit_scheduler.step()

# Plot alongside the other schedulers from the previous cell
plt.figure(figsize=(8, 4))
plt.plot(vit_lrs, label="Warmup+Cosine (ViT default)", lw=2, color='red')
plt.axvline(x=5, color='gray', linestyle='--', alpha=0.5, label='End of warmup')
plt.title("ViT Learning Rate Schedule: Warmup + Cosine")
plt.xlabel("Epoch"); plt.ylabel("Learning Rate"); plt.legend()
plt.show()

assert vit_lrs[0] < vit_lrs[4], "LR must increase during warmup"
assert vit_lrs[4] > vit_lrs[99], "LR must decrease after warmup"


### COMMON PITFALLS
*   **Applying StepLR locally:** If `StepLR` expects `step()` to be called per epoch, but you put it inside the batch iteration loop, your learning rate will plummet to zero infinitely faster than intended. Ensure you know if the scheduler maps to global steps or epochs.


## 3. Mixed Precision Training (AMP)

Storing a model in FP32 uses exactly 4 bytes per parameter. 
Modern Nvidia GPUs (Turing/Ampere+) have dedicated hardware called **Tensor Cores** that perform matrix multiplication in **FP16** natively, providing an instant 2-8X throughput boost and halving VRAM usage.

**The Underflow Problem:**
The smallest number FP16 can explicitly represent is $\sim 6 \times 10^{-5}$. 
In Deep Learning, gradients regularly approach $1 \times 10^{-8}$ midway through training. If you shove these into FP16, they underflow to exactly $0.0$. Backpropagation collapses.

**The Solution: GradScaler:**
We mathematically scale up the entire loss by a massive factor (e.g. $65536$) before computing the backward pass. This artificially rockets the tiny gradients safely into the physical float range of FP16. We perform backprop, and mathematically divide the weights back down by $65536$ during optimizer update.


In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: AMP Training Loop Framework
# -----------------------------------------------------
from torch.cuda.amp import autocast, GradScaler
import time

def simulate_mixed_precision_loop():
    # Only runs natively on CUDA
    if not torch.cuda.is_available():
        print("CUDA not available. Simulating AMP logic.")
        return

    model = nn.Linear(512, 10).cuda()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    
    # 1. Initialize the Scaler
    scaler = GradScaler()
    
    # Dummy data
    data = torch.randn(128, 512).cuda()
    target = torch.randn(128, 10).cuda()
    
    model.train()
    for step in range(5):
        optimizer.zero_grad()
        
        # 2. Enter AutoCast context (Forward pass uses FP16 automatically)
        with autocast():
            output = model(data)
            loss = loss_fn(output, target)
        
        # 3. Scale Loss, then Call Backward natively
        scaler.scale(loss).backward()
        
        # 4. Unscale and Step Optimizer
        scaler.step(optimizer)
        
        # 5. Update scale dynamically based on inf/NaN detection
        scaler.update()

# TEST IT
simulate_mixed_precision_loop()
print("AMP GradScaler logic successfully traced. VRAM effectively halved, Throughput doubled.")


## Gradient Accumulation

ViT-L/16 requires effective batch size 4096 for stable training. On a single 24GB GPU, max batch size might be 64. Gradient accumulation runs N steps without calling `optimizer.step()`, accumulating gradients until the effective batch size is reached.

In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: Gradient Accumulation
# -----------------------------------------------------
from torch.cuda.amp import autocast

def accumulation_step_demo():
    # Placeholder for objects
    model = nn.Linear(10, 1)
    optimizer = optim.Adam(model.parameters())
    scaler = GradScaler()
    loss_fn = nn.MSELoss()
    dataloader_sim = [(torch.randn(64, 10), torch.randn(64, 1))] * 128
    
    accumulation_steps = 64  # Simulate batch_size=64*64=4096
    
    optimizer.zero_grad()
    for step, (data, target) in enumerate(dataloader_sim):
        # We use a dummy try-except or check for CUDA to make the script portable
        context = autocast() if torch.cuda.is_available() else torch.no_grad()
        
        with context:
            output = model(data)
            loss = loss_fn(output, target)
            # DIVIDE loss by accumulation_steps so gradients have correct scale
            loss = loss / accumulation_steps
        
        if torch.cuda.is_available():
            scaler.scale(loss).backward()
        else:
            loss.backward()
        
        if (step + 1) % accumulation_steps == 0:
            if torch.cuda.is_available():
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            optimizer.zero_grad()

    print("Gradient accumulation cycle completed.")

# CRITICAL: If you do NOT divide the loss by accumulation_steps, your effective 
# gradient magnitude is accumulation_steps times larger than intended. 
# The optimizer step will behave as if you used an LR that is 64x too high.

accumulation_step_demo()


### COMMON PITFALLS
*   **Loss Scale NaN Explodes:** If your model creates `NaN`s internally, the `GradScaler` will catch it mathematically, reject the optimization step, and intelligently scale down the multiplier. But if your `autocast()` wraps loss functions that specifically demand FP32 accuracy (e.g. strict probability inversions), the gradients will shatter manually.


## 4. Evaluation Metrics Reference

A model is blind until it is evaluated mathematically. Here is the canonical comprehensive guide for ALL computer vision tracks.

| Domain | Metric | Meaning |
| :--- | :--- | :--- |
| **Classification** | Top-1 Accuracy | Standard % correct. |
| **Classification** | Top-5 Accuracy | Is the true class one of the model's top 5 guesses? (Crucial for 1000-class ImageNet). |
| **Classification** | Balanced Acc. | Explicit average of Recall on each class regardless of class imbalance in test set. |
| **Classification** | **MCC** | Matthews Correlation Coefficient. Handles extreme class imbalance organically (Scale -1 to +1). |
| **Detection** | mAP@0.5 | Mean Average Precision globally, considering an IoU overlap box of 50% physical territory. |
| **Detection** | mAP@[.5:.95] | Strict calculation averaging mAP at incredibly tight bounding box thresholds. |
| **Detection** | AR (Average Recall) | Maximum physical detection recall at various boundaries regardless of precision. |
| **Segmentation** | mIoU | Mean Intersection over Union. The strict percentage overlap of physical masks class by class. |
| **Segmentation** | Boundary IoU | Specifically penalizes models mapping coarse/blocky mask boundaries over fine details. |
| **Generation** | FID / IS | Fréchet Inception Distance / Inception Score. Measures representation diversity. |
| **Generation** | SSIM / PSNR | Structural Similarity / Peak Signal to Noise. Computes raw physical compression fidelity. |
| **Face/Metric** | Rank-1 / TAR | True Acceptance Rate at False Acceptance Rate thresholds. Controls strict security lockouts. |


In [ ]:
import numpy as np

# -----------------------------------------------------
# IMPLEMENTATION: Metrics computed from scratch
# -----------------------------------------------------

# 1. Intersection over Union (IoU) for Bounding Boxes / Segmentation
def compute_iou(boxA, boxB):
    # Determine intersection (x1, y1, x2, y2)
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)
    
    # Determine union
    boxAArea = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
    boxBArea = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)
    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

# 2. Balanced Accuracy
def balanced_accuracy(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes))
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    
    recalls = []
    for i in range(num_classes):
        hits = cm[i, i]
        total = cm[i, :].sum()
        if total > 0: recalls.append(hits / total)
        
    return np.mean(recalls)

# TEST IT
box_t = [50, 50, 150, 150]
box_p = [60, 60, 140, 140]
print(f"Bounding Box IoU: {compute_iou(box_t, box_p):.2f}")

y_t = [0, 0, 0, 0, 1]
y_p = [0, 0, 0, 1, 1] # Model got all majority class right (almost), and 1 minority class!
print(f"Balanced Accuracy: {balanced_accuracy(y_t, y_p):.2f}")


### COMMON PITFALLS
*   **mAP threshold mismatching:** Academic researchers love switching between `mAP@0.5` and `mAP@[0.5:0.95]` contextually to make their YOLO iterations look artificially superior in physics papers. Always align threshold scopes during benchmarking.
